# Using NATS

With `AMOORION_NATS` set, what methods change is published to
`<prefix>.methods.<name>`, and the daemon keeps what arrives on `<prefix>.in.>` in the
inbox. Off otherwise. This notebook has no server: it shows the subjects and how a
message is read. It has only been tried against a fake client so far.

In [1]:
from amoorion import Amoorion

# A sandbox for this whole notebook: secrets only from variables, the inbox and
# the dataset in a temporary folder. Nothing here can touch your real setup.
Amoorion.init()

PosixPath('/tmp/amoorion-rz4s5edg')

In [2]:
# Nothing connects here: no NATS server is needed to read this notebook.
from amoorion import nats as amoorion_nats

from amoorion.nats import in_subject, out_subject, to_message

# None: AMOORION_NATS isn't set, so amoorion stays away from NATS.
print(amoorion_nats.base.enabled())

# Results of methods go out on one subject; the daemon listens below the other.
print(
    out_subject("publish"),
    "|",
    in_subject(),
)

None
amoorion.methods.publish | amoorion.in.>


What arrives, as JSON or plain text:

In [3]:
# JSON with a `text` becomes a message with its author…
print(
    to_message(
        "amoorion.in.alerts",
        b'{"text": "disk full", "author": "monitor"}',
    )
)

# …and anything else is kept as plain text.
print(
    to_message(
        "amoorion.in.alerts",
        b"plain words",
    )
)

platform='nats:alerts' text='disk full' id=None url=None author=Account(platform='nats:alerts', handle='monitor', name=None) reply_to=None created_at=None
platform='nats:alerts' text='plain words' id=None url=None author=None reply_to=None created_at=None


With a server (`nats-server` alone is enough):

```sh
AMOORION_NATS=nats://localhost:4222 uv run amoorion daemon
nats pub amoorion.in.alerts '{"text": "disk full"}'
nats sub 'amoorion.methods.>'
```